# Lecture 8：Attention 与 Transformer

## 1. 为什么需要 Attention

传统的 Seq2Seq RNN 使用 Encoder 将输入序列 $$x_1,x_2,\dots,x_T$$ 编码为隐藏状态 $$h_t=f_W(x_t,h_{t-1}).$$ 然后通常使用 Encoder 最后的隐藏状态 $h_T$ 构造一个固定长度的上下文向量 $c$，Decoder 根据 $$s_t=g_U(y_{t-1},s_{t-1},c)$$ 逐步生成输出。

这种方法存在明显的**信息瓶颈（bottleneck）**：无论输入序列有多长，都必须将所有信息压缩进一个固定维度的向量 $c$。当输入序列非常长时，例如 $T=1000$，一个固定向量很难保存全部重要信息。

Attention 的核心思想是：

> Decoder 每生成一个输出时，都重新查看 Encoder 的所有隐藏状态，并根据当前需要决定重点关注哪些位置。

因此，不再对所有 Decoder 时间步使用同一个固定的 $c$，而是每个时间步都有自己的上下文向量 $$c_1,c_2,\dots,c_{T'}.$$

---

## 2. RNN 中的 Attention

设 Encoder 得到隐藏状态 $$h_1,h_2,\dots,h_T.$$ Decoder 在准备生成第 $t$ 个输出时，上一个 Decoder 隐藏状态为 $s_{t-1}$。

### 2.1 计算 Alignment Score

对每个 Encoder 隐藏状态 $h_i$，计算它与当前 Decoder 状态 $s_{t-1}$ 的相关程度：$$e_{t,i}=f_{\mathrm{att}}(s_{t-1},h_i).$$

其中 $e_{t,i}$ 称为 **alignment score（对齐分数）**。

此时可以把 $$q_t=s_{t-1}$$ 理解成当前的 **Query（查询向量）**，它表示：

> Decoder 当前需要从输入序列中寻找什么信息？

例如输入：

we see the sky

当 Decoder 准备生成意大利语 `vediamo` 时，当前 Query 可能主要需要寻找英语中的 `we` 和 `see`。

---

## 3. Attention Weight

得到所有 alignment score 后：$$e_{t,1},e_{t,2},\dots,e_{t,T},$$ 使用 Softmax 将其转换为 Attention 权重：$$a_{t,i}=\frac{\exp(e_{t,i})}{\sum_j\exp(e_{t,j})}.$$

因此：$$0<a_{t,i}<1,\qquad \sum_i a_{t,i}=1.$$

$a_{t,i}$ 表示：

> Decoder 在生成第 $t$ 个输出时，应该在多大程度上关注第 $i$ 个输入位置。

例如生成 `vediamo` 时，可能得到：$$a_t=[0.45,0.45,0.05,0.05].$$ 这表示模型主要关注 `we` 和 `see`。

---

## 4. Context Vector

使用 Attention 权重对所有 Encoder 隐藏状态进行加权求和：$$c_t=\sum_i a_{t,i}h_i.$$

$c_t$ 就是当前 Decoder 时间步从整个输入序列中提取出的信息。

然后 Decoder 使用：$$s_t=g_U(y_{t-1},s_{t-1},c_t)$$ 计算新的隐藏状态。

因此整个过程可以理解为：

生成 `vediamo`：
- Query：$s_0$
- 主要关注：`we`、`see`
- 得到：$c_1$

生成 `il`：
- Query：$s_1$
- 主要关注：`the`
- 得到：$c_2$

生成 `cielo`：
- Query：$s_2$
- 主要关注：`sky`
- 得到：$c_3$

Attention 的核心思想因此是：$$\boxed{\text{根据当前需求，从所有输入信息中动态选择并汇总最相关的部分}}.$$

---

## 5. $s_0$ 和 $y_0$

为了计算 Decoder 的第一个状态：$$s_1=g_U(y_0,s_0,c_1),$$ 必须预先给出 $s_0$ 和 $y_0$。

### 5.1 初始隐藏状态 $s_0$

$s_0$ 通常由 Encoder 的最终隐藏状态 $h_T$ 得到，例如：$$s_0=h_T,$$ 或者经过线性变换：$$s_0=W_sh_T+b_s.$$

它表示 Decoder 在开始生成之前，对整个输入序列的初始理解。

### 5.2 初始输入 $y_0$

$y_0$ 是人为定义的特殊 token：$$y_0=[\mathrm{START}].$$

它相当于告诉 Decoder：

> 现在开始生成输出序列。

例如训练目标可以写成：

[START] → vediamo → il → cielo → [STOP]

因此第一步使用：$$s_1=g_U(y_0,s_0,c_1).$$

---

## 6. 将 Attention 抽象成一般运算

脱离 RNN 后，可以把 Attention 看成一个一般的数据查询操作。

给定一个 Query：$$q\in\mathbb R^{D_Q},$$ 以及一组数据向量：$$X_1,X_2,\dots,X_N,$$ 首先计算每个数据向量与 Query 的相关性：$$e_i=f_{\mathrm{att}}(q,X_i).$$

然后：$$a=\operatorname{softmax}(e).$$

最后进行加权求和：$$y=\sum_i a_iX_i.$$

因此 Attention 可以概括成：$$\boxed{\text{Query}\rightarrow\text{计算相关性}\rightarrow\text{Softmax}\rightarrow\text{加权汇总}}.$$

---

## 7. 多个 Query

如果同时存在多个 Query，可以把它们堆叠成矩阵：$$Q\in\mathbb R^{N_Q\times D_Q}.$$

在前面的 RNN 翻译问题中，可以写成：$$Q=\begin{bmatrix}s_0^T\\s_1^T\\s_2^T\\\vdots\end{bmatrix}.$$

每一行对应一个 Decoder 时间步的 Query。

Encoder 隐藏状态也可以写成矩阵：$$X=\begin{bmatrix}h_1^T\\h_2^T\\\vdots\\h_T^T\end{bmatrix}.$$

如果 $D_Q=D_X$，则可以一次计算全部匹配关系：$$E=QX^T.$$

其中：$$E_{ij}=Q_i\cdot X_j$$ 表示第 $i$ 个 Query 与第 $j$ 个数据向量之间的相关程度。

---

## 8. 为什么引入 Query、Key、Value

在前面的简单 Attention 中，同一个数据向量 $X_i$ 同时承担两个角色：

1. 用来判断它与 Query 是否相关；
2. 如果被关注，它又直接提供输出信息。

Transformer 将这两个角色分开。

对于输入向量 $x_i$，定义：$$q_i=W_Qx_i,\qquad k_i=W_Kx_i,\qquad v_i=W_Vx_i.$$

其中：

- **Query $q$**：我想找什么；
- **Key $k$**：我具有什么特征，可以用来与你匹配；
- **Value $v$**：如果你选择关注我，我真正提供什么信息。

可以类比数据库：

- Query：搜索条件；
- Key：索引；
- Value：实际内容。

判断相关性时比较 Query 和 Key；真正汇总的是 Value。

---

## 9. Scaled Dot-Product Attention

Transformer 使用点积衡量 Query 与 Key 的相似度。

设：$$Q\in\mathbb R^{N_Q\times d_k},\qquad K\in\mathbb R^{N_K\times d_k},\qquad V\in\mathbb R^{N_K\times d_v}.$$

所有 Query 与 Key 之间的相似度可以一次计算：$$QK^T\in\mathbb R^{N_Q\times N_K}.$$

然后进行缩放：$$E=\frac{QK^T}{\sqrt{d_k}}.$$

再使用 Softmax：$$A=\operatorname{softmax}(E).$$

最后对 Value 加权求和：$$Y=AV.$$

因此 Transformer 中最重要的 Attention 公式是：$$\boxed{\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V}.$$

各部分含义：

$$QK^T\quad\rightarrow\quad\text{计算谁与谁相关}$$

$$\frac{1}{\sqrt{d_k}}\quad\rightarrow\quad\text{控制数值尺度}$$

$$\operatorname{softmax}\quad\rightarrow\quad\text{得到 Attention 权重}$$

$$AV\quad\rightarrow\quad\text{按照权重汇总 Value 信息}$$

---

## 10. 为什么除以 $\sqrt{d_k}$

设：$$q,k\in\mathbb R^{d_k}.$$ 它们的点积为：$$q^Tk=\sum_{i=1}^{d_k}q_ik_i.$$

随着 $d_k$ 增大，点积的数值尺度通常也会增大。

如果 Softmax 的输入绝对值非常大，例如：$$[20,-15,10],$$ 则 Softmax 会变得非常尖锐，接近 one-hot 分布，使梯度变小并增加优化困难。

因此除以 $\sqrt{d_k}$，使不同特征维数下点积的数值尺度更加稳定。

这一思想与 Xavier、Kaiming 初始化中控制数值方差的思想类似。

---

## 11. Self-Attention

如果 Query、Key、Value 全部来自**同一组输入向量** $X$：$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V,$$ 则：$$Y=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.$$

这就是 **Self-Attention（自注意力）**。

“Self”的含义是：

> 一组输入向量内部彼此查询、彼此交换信息。

例如文本：

The animal didn't cross the street because it was tired.

在处理 `it` 时，它可以通过 Self-Attention 直接关注 `animal`。

---

## 12. Self-Attention 与 RNN 的区别

RNN 中，信息沿时间顺序逐步传播：$$x_1\rightarrow h_1\rightarrow h_2\rightarrow\cdots\rightarrow h_{100}.$$

如果第 100 个 token 需要使用第 1 个 token 的信息，需要经过许多中间隐藏状态。

Self-Attention 中，第 1 个和第 100 个 token 可以在一层中直接产生联系。

因此：

- RNN 的长距离信息传播路径随序列长度增长；
- Self-Attention 可以在一层内实现全局信息交互；
- RNN 的时间步之间存在顺序依赖，不容易完全并行；
- Self-Attention 主要由矩阵乘法构成，非常适合 GPU 并行计算。

但标准 Self-Attention 需要计算：$$QK^T\in\mathbb R^{N\times N},$$ 因此计算量和显存消耗会随序列长度 $N$ 近似按 $N^2$ 增长。

---

## 13. Positional Encoding

普通 Self-Attention 本身不知道输入的顺序。

例如：

dog bites man

与：

man bites dog

如果只把它们看成一组向量，Self-Attention 本身没有任何机制知道哪个 token 在前、哪个 token 在后。

因此加入 **Positional Encoding（位置编码）**：$$z_i=x_i+p_i.$$

其中：

- $x_i$：token 自身的语义表示；
- $p_i$：第 $i$ 个位置的位置表示。

这样模型同时知道：

1. token 是什么；
2. token 位于哪里。

---

## 14. Masked Self-Attention

语言模型执行 Next-Token Prediction 时，第 $t$ 个 token 不能看到未来的 token。

例如训练：

Attention → is

Attention is → all

Attention is all → you

当模型预测 `is` 时，不能提前看到 `all` 和 `you`。

因此对相似度矩阵进行 Mask。

若：$$E=\begin{bmatrix}e_{11}&e_{12}&e_{13}\\e_{21}&e_{22}&e_{23}\\e_{31}&e_{32}&e_{33}\end{bmatrix},$$ 则将未来位置设为 $-\infty$：$$E'=\begin{bmatrix}e_{11}&-\infty&-\infty\\e_{21}&e_{22}&-\infty\\e_{31}&e_{32}&e_{33}\end{bmatrix}.$$

由于：$$e^{-\infty}=0,$$ 经过 Softmax 后未来 token 的 Attention 权重就变成 $0$。

这称为：

- Masked Self-Attention；
- Causal Self-Attention。

其核心约束是：$$\boxed{\text{第 }i\text{ 个 token 只能关注自己以及之前的 token}}.$$

---

## 15. Multi-Head Attention

单个 Attention Head 只有一套：$$W_Q,W_K,W_V.$$

但序列中往往同时存在许多不同类型的关系。

例如：

The small red car that I bought yesterday is fast.

可能存在：

- `car ↔ red`：属性关系；
- `car ↔ small`：属性关系；
- `car ↔ bought`：语法关系；
- `car ↔ is`：主谓关系。

因此使用多个 Attention Head 并行计算。

第 $h$ 个 Head 为：$$\operatorname{head}_h=\operatorname{Attention}(Q_h,K_h,V_h).$$

每个 Head 都有自己独立的参数：$$W_Q^{(h)},\qquad W_K^{(h)},\qquad W_V^{(h)}.$$

计算完成后，将所有 Head 拼接：$$H=\operatorname{Concat}(\operatorname{head}_1,\dots,\operatorname{head}_m),$$ 再通过输出投影：$$Y=HW_O.$$

直觉上：

> 不同 Attention Head 可以学习不同类型的关系。

这和 CNN 中多个输出通道有一定相似性：

- CNN 使用多个通道，让不同通道有机会学习不同视觉特征；
- Transformer 使用多个 Head，让不同 Head 有机会学习不同 token 关系。

但并不保证不同 Head 一定学到完全不同的信息。

---

## 16. Attention 与 Transformer 的区别

**Attention** 是一种信息查询与聚合运算：$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.$$

**Transformer** 是由多个组件组合成的完整神经网络架构，其中包括：

- Multi-Head Self-Attention；
- MLP / FFN；
- Residual Connection；
- Layer Normalization。

因此：$$\boxed{\text{Attention 是组件，Transformer 是完整网络架构}}.$$

---

## 17. Transformer Block

一个经典 Transformer Block 可以抽象为：

Input

↓

Multi-Head Self-Attention

↓

Residual Connection

↓

LayerNorm

↓

MLP / FFN

↓

Residual Connection

↓

LayerNorm

↓

Output

数学上可以写成：$$X'=\operatorname{LayerNorm}\left(X+\operatorname{MHA}(X)\right),$$ $$Y=\operatorname{LayerNorm}\left(X'+\operatorname{MLP}(X')\right).$$

其中最重要的分工是：

- **Self-Attention**：负责不同 token 之间交换信息；
- **MLP**：负责每个 token 内部进一步处理特征；
- **Residual Connection**：帮助训练深层网络；
- **LayerNorm**：稳定特征数值尺度。

---

## 18. Transformer 中的 MLP / FFN

Attention 得到每个 token 的新表示后，对每个 token 独立应用相同的 MLP。

经典结构为：$$D\rightarrow4D\rightarrow D.$$

即：$$x\rightarrow W_1x+b_1\rightarrow\sigma\rightarrow W_2x+b_2.$$

它通常称为：

- MLP；
- Feed-Forward Network（FFN）。

需要特别注意：

> MLP 本身不会让不同 token 交换信息。

假设有：$$x_1,x_2,x_3,$$ MLP 实际执行的是：$$x_1\rightarrow\operatorname{MLP}(x_1),$$ $$x_2\rightarrow\operatorname{MLP}(x_2),$$ $$x_3\rightarrow\operatorname{MLP}(x_3).$$

不同 token 真正发生信息交互的位置是 Self-Attention。

因此可以总结为：$$\boxed{\text{Attention：token 之间交流；MLP：token 内部加工}}.$$

---

## 19. Residual Connection

Transformer 中大量使用残差连接：$$Y=X+F(X).$$

例如 Attention 部分：$$X'=X+\operatorname{Attention}(X).$$

MLP 部分：$$Y=X'+\operatorname{MLP}(X').$$

Residual Connection 的主要作用包括：

1. 保留原始信息；
2. 改善梯度传播；
3. 使网络容易学习恒等映射；
4. 支持堆叠非常深的 Transformer Block。

这一思想与 ResNet 中的残差连接基本相同。

---

## 20. Layer Normalization

Transformer 通常使用 **LayerNorm**，而不是 CNN 中常见的 BatchNorm。

设一个 token 的表示为：$$x_i\in\mathbb R^D.$$ LayerNorm 在该 token 自己的 $D$ 个特征上计算均值和方差：$$\hat x_i=\frac{x_i-\mu_i}{\sqrt{\sigma_i^2+\epsilon}}.$$

因此 LayerNorm 不需要依赖 batch 中其他样本的统计量。

这对于序列模型尤其方便，因为：

- 不同样本的序列长度可能不同；
- batch size 可能变化；
- 推理时也可能只有一个样本。

---

## 21. Pre-Norm Transformer

早期 Transformer 常采用类似：$$\operatorname{LayerNorm}(X+F(X)).$$

现代 Transformer 更常见 **Pre-Norm** 结构，即先进行 LayerNorm，再进入 Attention 或 MLP：$$X'=X+\operatorname{Attention}(\operatorname{LN}(X)),$$ $$Y=X'+\operatorname{MLP}(\operatorname{LN}(X')).$$

结构可以理解为：

LayerNorm

↓

Self-Attention

↓

Residual

↓

LayerNorm

↓

MLP

↓

Residual

Pre-Norm 通常能够让非常深的 Transformer 训练更加稳定。

---

## 22. 完整 Transformer

Transformer 本质上就是堆叠许多结构相同的 Transformer Block：

Input Embedding

↓

Positional Encoding

↓

Transformer Block

↓

Transformer Block

↓

Transformer Block

↓

...

↓

Task Head

不同 Transformer Block 有各自独立的参数。

Transformer 具有很强的可扩展性，因为核心计算大部分都是矩阵乘法，因此非常适合 GPU / TPU 并行计算。

---

## 23. Transformer 如何用于语言模型

设词表大小为 $V$，模型维度为 $D$。

首先使用 Embedding Matrix：$$E\in\mathbb R^{V\times D}$$ 将离散 token ID 转换为 $D$ 维向量。

例如：

Attention is all you

首先变成：$$x_1,x_2,x_3,x_4,\qquad x_i\in\mathbb R^D.$$

然后加入位置编码：$$z_i=x_i+p_i.$$

再送入许多 Transformer Block。

对于 Next-Token Prediction，需要使用 Masked Self-Attention，使第 $t$ 个位置只能看到：$$1,2,\dots,t.$$

最后，每个位置得到隐藏状态：$$h_t\in\mathbb R^D.$$

再通过线性层投影到整个词表：$$z_t=W_oh_t,$$ 其中：$$W_o\in\mathbb R^{V\times D}.$$

经过 Softmax 得到下一个 token 的概率：$$p_t=\operatorname{softmax}(z_t).$$

训练目标例如：

Attention → is

Attention is → all

Attention is all → you

Attention is all you → need

通常使用 Cross-Entropy Loss。

---

## 24. RNN 与 Transformer 语言模型的区别

两者的训练目标可以完全相同，例如都是 Next-Token Prediction。

主要区别在于上下文信息如何传播。

RNN：$$h_t=f(x_t,h_{t-1}).$$ 因此上下文通过隐藏状态按时间顺序递归传播。

Transformer：$$Y=\operatorname{SelfAttention}(X).$$ 每个 token 可以直接访问允许范围内的其他 token。

因此：$$\boxed{\text{RNN 使用递归 Hidden State 传递上下文；Transformer 使用 Self-Attention 直接交换上下文信息}}.$$

---

## 25. Vision Transformer（ViT）

Transformer 原本接收的是“一组向量”，因此处理图片时，需要先把图片转换成一组向量。

例如输入图片：$$224\times224\times3.$$

将其切分成大小为：$$16\times16$$ 的 Patch。

每行 Patch 数为：$$224/16=14.$$

因此总 Patch 数：$$14\times14=196.$$

每个 Patch 包含：$$16\times16\times3=768$$ 个数。

将每个 Patch Flatten：$$\mathbb R^{16\times16\times3}\rightarrow\mathbb R^{768},$$ 再进行线性映射：$$768\rightarrow D.$$

最终整张图片变成：$$x_1,x_2,\dots,x_{196},\qquad x_i\in\mathbb R^D.$$

这些 Patch Vector 就类似 NLP 中的 token。

---

## 26. Patch Embedding 与卷积的关系

“切 Patch + Flatten + Linear Projection”实际上等价于使用一个卷积层。

例如 Patch 大小为 $16\times16$，可以使用：

$$\mathrm{Conv2d}(3,D,\mathrm{kernel\ size}=16,\mathrm{stride}=16).$$

其作用是：

- 每次读取一个 $16\times16$ 区域；
- 不同 Patch 之间不重叠；
- 将每个 Patch 映射成一个 $D$ 维向量。

因此 ViT 的 Patch Embedding 可以理解成一种特殊的大步长卷积。

---

## 27. ViT 中的位置编码

Self-Attention 本身不知道 Patch 在原图中的二维位置，因此需要加入 Positional Encoding。

设第 $i$ 个 Patch 的表示为 $x_i$，则：$$z_i=x_i+p_i.$$

其中 $p_i$ 表示该 Patch 在原图中的空间位置。

这样 Transformer 才能区分：

- 左上角 Patch；
- 中央 Patch；
- 右下角 Patch。

---

## 28. ViT 是否需要 Mask

图像分类与语言生成不同。

语言模型中，第 $t$ 个 token 不能看到未来 token，因此需要 Causal Mask。

但图像分类时，没有所谓“未来 Patch”。

因此 ViT 通常**不使用 Causal Mask**：

> 每个 Patch 可以直接关注图像中的所有其他 Patch。

所以第一层 Self-Attention 就可以建立全局图像关系。

---

## 29. ViT 的完整流程

ViT 可以概括为：

Image

↓

Split into Patches

↓

Patch Embedding

↓

Add Positional Encoding

↓

Transformer Blocks

↓

Pooling / Classification Representation

↓

Linear Classification Head

↓

Class Scores

因此：$$\boxed{\text{Image}\rightarrow\text{Patch Tokens}\rightarrow\text{Transformer}\rightarrow\text{Classification}}.$$

---

## 30. CNN 与 Transformer 的核心区别

CNN 的核心视觉归纳偏置是：$$\boxed{\text{局部性 + 参数共享 + 平移等变性}}.$$

一个 $3\times3$ 卷积只能直接看到附近区域。

随着网络逐渐加深：

局部特征

↓

更大的局部结构

↓

物体部件

↓

全局语义

因此 CNN 通常是逐步扩大感受野。

Transformer 的 Self-Attention 则允许：

> 第一层中每个 token 就能够直接与所有其他 token 发生信息交互。

因此可以概括为：

CNN：

局部 → 更大局部 → 更大区域 → 全局

Transformer：

第一层即可建立全局关系

---

## 31. CNN 与 Transformer 的归纳偏置

CNN 天生假设：

1. 图像的重要结构通常是局部的；
2. 同一种模式可能出现在不同空间位置；
3. 不同位置可以共享同一组卷积核。

因此 CNN 拥有较强的视觉归纳偏置。

Transformer 的结构假设更弱。

Self-Attention 允许任意 Patch 与任意其他 Patch 建立联系，所以它更加灵活，但也意味着：

- 往往需要更多数据；
- 需要额外提供位置信息；
- 标准 Self-Attention 的计算和显存成本随 token 数量平方增长。

因此不能简单理解为：$$\text{Transformer}>\text{CNN}.$$ 更准确地说：

> CNN 与 Transformer 使用了不同的归纳偏置。

---

## 32. 从 RNN 到 Transformer 的发展逻辑

Lecture 8 最重要的并不是记住很多结构，而是理解下面这条发展链：

Seq2Seq RNN

↓

整个输入被压缩进固定 Context Vector

↓

出现信息瓶颈

↓

Attention

↓

每个 Decoder 时间步动态选择相关输入

↓

将 Attention 抽象成一般查询操作

↓

Query / Key / Value

↓

Scaled Dot-Product Attention

↓

Self-Attention

↓

Multi-Head Self-Attention

↓

加入 MLP、Residual Connection、LayerNorm

↓

Transformer Block

↓

堆叠多个 Transformer Block

↓

Transformer

↓

文本 Token → Language Model

图像 Patch → Vision Transformer

---

## 33. Attention 最重要的公式

整个 Lecture 8 最重要的公式是：$$\boxed{\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V}.$$

不应该只背公式，而应该理解其中每一部分。

### Query

$$\boxed{Q=\text{我想寻找什么}}$$

### Key

$$\boxed{K=\text{我有什么特征供你判断是否相关}}$$

### Value

$$\boxed{V=\text{如果你关注我，我实际提供什么信息}}$$

### $QK^T$

$$\boxed{QK^T=\text{计算每个 Query 与每个 Key 的相关程度}}$$

### Softmax

$$\boxed{\operatorname{softmax}=\text{将相关程度转换成 Attention 权重}}$$

### 与 $V$ 相乘

$$\boxed{AV=\text{按照 Attention 权重汇总真正的信息}}$$

因此 Attention 的完整直觉是：$$\boxed{\text{提出查询}\rightarrow\text{寻找相关信息}\rightarrow\text{分配关注权重}\rightarrow\text{汇总信息}}.$$

---

## 34. Transformer Block 最重要的理解

Transformer Block 可以分成两类操作。

### Self-Attention：信息交流

$$\boxed{\text{不同 token 之间交换信息}}$$

它回答：

> 我应该从其他 token 中获取哪些信息？

### MLP：信息加工

$$\boxed{\text{对每个 token 自己的特征进行进一步非线性变换}}$$

它回答：

> 我已经获得这些信息以后，应该如何进一步处理这些特征？

### Residual Connection

$$\boxed{\text{保留原信息，并改善深层网络的梯度传播}}$$

### LayerNorm

$$\boxed{\text{稳定每个 token 的特征尺度，使训练更加稳定}}$$

因此整个 Transformer Block 可以直观理解为：$$\boxed{\text{Attention 负责交流，MLP 负责加工，Residual 与 LayerNorm 负责让深层网络稳定训练}}.$$

---

## 35. Lecture 8 核心结论

1. Attention 最初是为了解决 Seq2Seq RNN 的固定 Context Vector 信息瓶颈。
2. Attention 的本质是根据 Query 动态选择并汇总相关信息。
3. Query 表示“当前需要寻找什么信息”。
4. Key 用于与 Query 判断相关性。
5. Value 是真正被加权汇总的信息。
6. Scaled Dot-Product Attention 的核心公式为：$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.$$
7. Self-Attention 表示 Query、Key、Value 都来自同一组输入。
8. Self-Attention 允许任意两个 token 在一层中直接交换信息。
9. Self-Attention 本身不知道顺序，因此需要 Positional Encoding。
10. 自回归语言模型需要 Masked / Causal Self-Attention，防止看到未来信息。
11. Multi-Head Attention 使用多套独立参数，使模型能够在多个表示空间中学习不同关系。
12. Attention 是一种运算，Transformer 是一种完整网络架构。
13. Transformer Block 的核心由 Self-Attention、MLP、Residual Connection 和 LayerNorm 构成。
14. Self-Attention 负责 token 之间的信息交流，MLP 负责 token 内部的信息加工。
15. Transformer 可以通过堆叠大量结构相同的 Block 扩展到非常大的模型。
16. 语言模型把文本转换为 token 向量，再使用 Masked Transformer 进行 Next-Token Prediction。
17. ViT 将图像切成 Patch，把每个 Patch 当作一个 token 交给 Transformer。
18. CNN 具有较强的局部性和平移等变归纳偏置，而 Transformer 更强调全局关系建模。
19. Transformer 并不是简单“比 CNN 更先进”，而是采用了不同的结构假设。
20. 从 Attention 到 Transformer 的核心思想始终没有改变：$$\boxed{\text{根据当前表示动态决定应该从其他位置获取哪些信息}}.$$